# Final Roadworks ML Notebook — Google Colab

Downloads the Kaggle dataset `sobhanmoosavi/us-road-construction-and-closures`, explores it with Plotly, and trains optional models if suitable columns exist.

**Research limitation:** the dataset is US roadworks, not Berlin-specific mobility data. It does not directly measure secondary mobility impacts such as travel-time changes, traffic volumes, detour flows, or transit delays. Do not present its results as validated predictions of Berlin mobility impacts.

Run cells from top to bottom. After a runtime restart, run all cells again.


## 1. Install packages

In [ ]:
%pip -q install kaggle plotly scikit-learn joblib

## 2. Download the dataset from Kaggle

Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets and enable notebook access. If unavailable, the cell asks for them. Keep your API key private.


In [ ]:
import os, zipfile, subprocess
from pathlib import Path
from getpass import getpass

DATASET = "sobhanmoosavi/us-road-construction-and-closures"
DATA_DIR = Path("/content/roadworks_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

username = secret("KAGGLE_USERNAME") or os.environ.get("KAGGLE_USERNAME") or input("Kaggle username: ").strip()
api_key = secret("KAGGLE_KEY") or os.environ.get("KAGGLE_KEY") or getpass("Kaggle API key (hidden): ").strip()
if not username or not api_key:
    raise ValueError("Kaggle username and API key are required.")

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = api_key
result = subprocess.run(
    ["kaggle", "datasets", "download", "-d", DATASET, "-p", str(DATA_DIR), "--force"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Kaggle download failed. Check credentials, internet, dataset slug, and Kaggle terms.")

zips = list(DATA_DIR.glob("*.zip"))
if not zips:
    raise FileNotFoundError("No ZIP downloaded.")
for z in zips:
    with zipfile.ZipFile(z) as archive:
        archive.extractall(DATA_DIR)

csv_files = sorted(DATA_DIR.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV files found in downloaded archive.")
for i, p in enumerate(csv_files):
    print(i, p, f"({p.stat().st_size:,} bytes)")


## 3. Load and inspect the dataset

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go

CSV_INDEX = 0  # Change if the desired CSV is another item in the list above.
if CSV_INDEX >= len(csv_files):
    raise IndexError("CSV_INDEX is outside the available CSV list.")
csv_path = csv_files[CSV_INDEX]
df = pd.read_csv(csv_path, low_memory=False)

print("File:", csv_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [(df[c].isna().mean() * 100).round(2) for c in df.columns]
})
display(schema.sort_values("missing_percent", ascending=False))


## 4. Detect dates and potential targets

In [ ]:
def find_column(columns, candidates):
    lower = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    for col in columns:
        if any(candidate.lower() in str(col).strip().lower() for candidate in candidates):
            return col
    return None

START_COL = find_column(df.columns, ["start_time", "start date", "start_date", "starttime", "begin_time"])
END_COL = find_column(df.columns, ["end_time", "end date", "end_date", "endtime", "finish_time"])
DATE_COL = START_COL or find_column(df.columns, ["timestamp", "event_date", "date", "datetime", "time"])
TYPE_COL = find_column(df.columns, ["closure_type", "construction_type", "roadwork_type", "event_type", "type"])

print("Start column:", START_COL)
print("End column:", END_COL)
print("Date column:", DATE_COL)
print("Possible type target:", TYPE_COL)
print("If detection is wrong, manually set START_COL, END_COL, DATE_COL, or TYPE_COL to the exact column name.")


## 5. Parse dates and create features

In [ ]:
for col in set(c for c in [START_COL, END_COL, DATE_COL] if c is not None):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

if START_COL and END_COL and START_COL in df.columns and END_COL in df.columns:
    df["duration_hours"] = (df[END_COL] - df[START_COL]).dt.total_seconds() / 3600
    df.loc[(df["duration_hours"] < 0) | (df["duration_hours"] > 24 * 365), "duration_hours"] = np.nan
    print("Created duration_hours.")
else:
    print("No duration target: both start and end columns were not detected.")

if DATE_COL and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce", utc=True)
    df["event_year"] = df[DATE_COL].dt.year
    df["event_month"] = df[DATE_COL].dt.month
    df["event_dayofweek"] = df[DATE_COL].dt.dayofweek
    df["event_hour"] = df[DATE_COL].dt.hour
    print("Created calendar features.")
else:
    print("No usable date column detected.")


## 6. Exploratory data analysis with Plotly

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False).head(25)
missing_df = missing.rename_axis("column").reset_index(name="missing_percent")
fig = px.bar(missing_df, x="column", y="missing_percent", title="Columns with most missing values")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

for col in df.select_dtypes(include=np.number).columns[:8]:
    px.histogram(df, x=col, nbins=40, title=f"Distribution of {col}").show()

for col in df.select_dtypes(include=["object", "category", "bool"]).columns[:5]:
    counts = df[col].astype("string").fillna("Missing").value_counts().head(20).rename_axis(col).reset_index(name="count")
    fig = px.bar(counts, x=col, y="count", title=f"Most common values: {col}")
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


## 7. Monthly roadwork records

In [ ]:
# Self-contained: this cell checks df and detects the date column if necessary.
if "df" not in globals():
    print("Dataset not loaded. Run Sections 2 and 3 first.")
else:
    candidates = ["start_time", "start date", "start_date", "starttime", "timestamp", "event_date", "date", "datetime", "time"]
    date_col = globals().get("DATE_COL", None)
    if date_col not in df.columns:
        date_col = next((c for c in df.columns if str(c).strip().lower() in candidates), None)

    if date_col is None:
        print("No date column detected. Available columns:", list(df.columns))
    else:
        dated = df.copy()
        dated[date_col] = pd.to_datetime(dated[date_col], errors="coerce", utc=True)
        dated = dated.dropna(subset=[date_col])
        if dated.empty:
            print(f"No valid dates in {date_col!r}.")
        else:
            dated = dated.set_index(date_col).sort_index()
            monthly_counts = dated.resample("MS").size().rename("events").reset_index()
            fig = px.line(monthly_counts, x=date_col, y="events", markers=True, title="Roadwork records per month")
            fig.update_layout(xaxis_title="Month", yaxis_title="Record count")
            fig.show()
            display(monthly_counts.head())


## 8. Classification model (optional)

Predicts a detected type/closure label, **not mobility impact**. Training is capped at 30,000 rows to reduce runtime.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

classification_model = None
classification_metrics = None
classification_test_data = None

if TYPE_COL and TYPE_COL in df.columns:
    work = df.dropna(subset=[TYPE_COL]).copy()
    work[TYPE_COL] = work[TYPE_COL].astype(str)
    counts = work[TYPE_COL].value_counts()
    work = work[work[TYPE_COL].isin(counts[counts >= 2].index)].copy()

    if len(work) < 20 or work[TYPE_COL].nunique() < 2:
        print("Not enough examples/classes for classification.")
    else:
        if len(work) > 30000:
            work = work.sample(30000, random_state=42)
        y = work[TYPE_COL]
        drop_cols = {TYPE_COL}
        if END_COL:
            drop_cols.add(END_COL)
        for col in work.columns:
            low = str(col).lower()
            if low.endswith("_id") or low in {"id", "event_id", "case_id", "record_id"}:
                drop_cols.add(col)
            if pd.api.types.is_datetime64_any_dtype(work[col]):
                drop_cols.add(col)
        X = work.drop(columns=[c for c in drop_cols if c in work.columns], errors="ignore")
        X = X.loc[:, X.notna().any()]
        X = X.loc[:, X.nunique(dropna=True) > 1]
        for col in X.select_dtypes(exclude=np.number).columns.tolist():
            if X[col].nunique(dropna=True) > 100:
                X = X.drop(columns=[col])

        if X.shape[1] == 0:
            print("No usable predictors remain.")
        else:
            if DATE_COL and DATE_COL in work and work[DATE_COL].notna().sum() > 10:
                order = work[DATE_COL].sort_values().index
                X, y = X.loc[order], y.loc[order]
                cut = int(len(X) * 0.8)
                X_train, X_test = X.iloc[:cut], X.iloc[cut:]
                y_train, y_test = y.iloc[:cut], y.iloc[cut:]
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.2, random_state=42, stratify=y
                )

            nums = X_train.select_dtypes(include=np.number).columns.tolist()
            cats = X_train.select_dtypes(exclude=np.number).columns.tolist()
            transformers = []
            if nums:
                transformers.append(("num", SimpleImputer(strategy="median"), nums))
            if cats:
                cat_pipe = Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))
                ])
                transformers.append(("cat", cat_pipe, cats))

            classification_model = Pipeline([
                ("preprocessing", ColumnTransformer(transformers)),
                ("model", RandomForestClassifier(
                    n_estimators=60, max_depth=15, max_features="sqrt",
                    min_samples_leaf=3, class_weight="balanced_subsample",
                    random_state=42, n_jobs=-1
                ))
            ])
            classification_model.fit(X_train, y_train)
            pred = classification_model.predict(X_test)
            classification_metrics = {
                "accuracy": accuracy_score(y_test, pred),
                "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
                "test_rows": len(y_test)
            }
            classification_test_data = (X_test, y_test)
            print(classification_metrics)
            print(classification_report(y_test, pred, zero_division=0))
            ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45)
            plt.tight_layout()
            plt.show()
else:
    print("No type/closure target detected. Check TYPE_COL.")


## 9. Duration regression (optional)

Runs only if start and end timestamps were found. Training is capped at 30,000 records and uses a smaller Random Forest to avoid very long execution.


In [ ]:
regression_model = None
regression_metrics = None
regression_test_data = None

if "duration_hours" not in df.columns:
    print("duration_hours unavailable. Check START_COL and END_COL.")
else:
    work = df.dropna(subset=["duration_hours"]).copy()
    work = work[(work["duration_hours"] >= 0) & (work["duration_hours"] <= 24 * 365)]

    if len(work) < 30:
        print("Not enough valid duration records for regression.")
    else:
        if len(work) > 30000:
            work = work.sample(30000, random_state=42)
        y = work["duration_hours"].astype(float)
        drop_cols = {"duration_hours"}
        if END_COL:
            drop_cols.add(END_COL)
        for col in work.columns:
            low = str(col).lower()
            if low.endswith("_id") or low in {"id", "event_id", "case_id", "record_id"}:
                drop_cols.add(col)
            if pd.api.types.is_datetime64_any_dtype(work[col]):
                drop_cols.add(col)
        X = work.drop(columns=[c for c in drop_cols if c in work.columns], errors="ignore")
        X = X.loc[:, X.notna().any()]
        X = X.loc[:, X.nunique(dropna=True) > 1]
        for col in X.select_dtypes(exclude=np.number).columns.tolist():
            if X[col].nunique(dropna=True) > 100:
                X = X.drop(columns=[col])

        if X.shape[1] == 0:
            print("No usable predictors remain for regression.")
        else:
            if DATE_COL and DATE_COL in work and work[DATE_COL].notna().sum() > 10:
                order = work[DATE_COL].sort_values().index
                X, y = X.loc[order], y.loc[order]
                cut = int(len(X) * 0.8)
                X_train, X_test = X.iloc[:cut], X.iloc[cut:]
                y_train, y_test = y.iloc[:cut], y.iloc[cut:]
            else:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.2, random_state=42
                )

            nums = X_train.select_dtypes(include=np.number).columns.tolist()
            cats = X_train.select_dtypes(exclude=np.number).columns.tolist()
            transformers = []
            if nums:
                transformers.append(("num", SimpleImputer(strategy="median"), nums))
            if cats:
                cat_pipe = Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))
                ])
                transformers.append(("cat", cat_pipe, cats))

            regression_model = Pipeline([
                ("preprocessing", ColumnTransformer(transformers)),
                ("model", RandomForestRegressor(
                    n_estimators=50, max_depth=12, max_features="sqrt",
                    min_samples_leaf=5, random_state=42, n_jobs=-1
                ))
            ])
            regression_model.fit(X_train, y_train)
            pred = regression_model.predict(X_test)
            from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
            regression_metrics = {
                "MAE_hours": mean_absolute_error(y_test, pred),
                "RMSE_hours": mean_squared_error(y_test, pred) ** 0.5,
                "R2": r2_score(y_test, pred),
                "test_rows": len(y_test)
            }
            regression_test_data = (X_test, y_test)
            print(regression_metrics)
            fig = px.scatter(
                x=y_test, y=pred,
                labels={"x": "Actual duration (hours)", "y": "Predicted duration (hours)"},
                title="Actual vs predicted roadwork duration"
            )
            lo, hi = float(min(np.min(y_test), np.min(pred))), float(max(np.max(y_test), np.max(pred)))
            fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="Perfect prediction"))
            fig.show()


## 10. Feature importance (safe if models were not trained)

In [ ]:
def plot_feature_importance(model, label):
    if model is None:
        print(f"{label}: model not trained; skipping.")
        return
    try:
        prep = model.named_steps["preprocessing"]
        estimator = model.named_steps["model"]
        names = prep.get_feature_names_out()
        vals = estimator.feature_importances_
        importance = pd.DataFrame({"feature": names, "importance": vals})
        importance = importance.sort_values("importance", ascending=False).head(20)
        display(importance)
        fig = px.bar(
            importance.sort_values("importance"), x="importance", y="feature",
            orientation="h", title=f"Top 20 feature importance — {label}"
        )
        fig.show()
    except Exception as exc:
        print(f"Could not calculate {label} feature importance: {exc}")

plot_feature_importance(classification_model, "Classification")
plot_feature_importance(regression_model, "Duration regression")


## 11. Monthly seasonal-naive baseline

In [ ]:
# Shows recent monthly counts alongside the same-month-last-year baseline.
if "df" not in globals():
    print("Dataset not loaded. Run the loading cells first.")
else:
    candidates = ["start_time", "start date", "start_date", "starttime", "timestamp", "event_date", "date", "datetime", "time"]
    date_col = globals().get("DATE_COL", None)
    if date_col not in df.columns:
        date_col = next((c for c in df.columns if str(c).strip().lower() in candidates), None)

    if date_col is None:
        print("No date column detected. Available columns:", list(df.columns))
    else:
        dated = df.copy()
        dated[date_col] = pd.to_datetime(dated[date_col], errors="coerce", utc=True)
        dated = dated.dropna(subset=[date_col])
        if dated.empty:
            print("No valid dates found.")
        else:
            dated = dated.set_index(date_col).sort_index()
            monthly = dated.resample("MS").size().rename("events").to_frame().asfreq("MS", fill_value=0)
            if len(monthly) < 24:
                print("At least 24 monthly observations are needed for this baseline.")
            else:
                monthly["seasonal_naive"] = monthly["events"].shift(12)
                recent = monthly.tail(12)
                display(recent)
                fig = go.Figure()
                fig.add_trace(go.Scatter(x=monthly.index, y=monthly["events"], mode="lines", name="Observed"))
                fig.add_trace(go.Scatter(x=recent.index, y=recent["seasonal_naive"], mode="lines+markers", name="Same month previous year"))
                fig.update_layout(title="Monthly Roadwork Records", xaxis_title="Month", yaxis_title="Record count")
                fig.show()


## 12. Save models and cleaned data

In [ ]:
import joblib
OUTPUT_DIR = Path("/content/roadworks_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if classification_model is not None:
    joblib.dump(classification_model, OUTPUT_DIR / "classification_model.joblib")
if regression_model is not None:
    joblib.dump(regression_model, OUTPUT_DIR / "duration_regression_model.joblib")

df.to_csv(OUTPUT_DIR / "roadworks_cleaned.csv", index=False)
print("Saved output files:", [p.name for p in OUTPUT_DIR.iterdir()])


## 13. Thesis interpretation checklist

- Describe the source and geography accurately: US roadworks, not Berlin.
- Report record counts, missing-data handling, target definition, split method, and model metrics.
- Feature importance indicates predictive association, not causation.
- Do not describe roadwork type or duration as a measured secondary mobility impact.
- Berlin-specific claims require Berlin roadwork records and aligned mobility outcomes, such as traffic counts, travel-time changes, detour flows, or public-transport delays.
